# Tokenization & Embeddings - End-to-End Demo

This notebook demonstrates the complete workflow:
1. Tokenization with different models
2. Generating embeddings
3. Visualizing with PCA

In [ ]:
# Setup - add src to path
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt

from token_emb import (
    TokenizerWrapper,
    EmbeddingProvider,
    PCAVisualizer,
)

## 1. Tokenization Examples

In [ ]:
# Sample texts
texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning is transforming how we build software.",
    "Natural language processing enables machines to understand text.",
    "Deep learning models require large amounts of data and compute.",
    "Tokenization is the first step in processing text data.",
]

# Count tokens with different models
models = ["gpt-4", "bert-base-uncased"]

for model in models:
    print(f"\n{'='*50}")
    print(f"Model: {model}")
    print(f"{'='*50}")
    
    tokenizer = TokenizerWrapper(model)
    for text in texts:
        count = tokenizer.count(text)
        print(f"{count:3d} tokens | {text[:50]}...")

## 2. Generate Embeddings

In [ ]:
# Initialize embedding provider
provider = EmbeddingProvider("sentence-transformers/all-MiniLM-L6-v2")

print(f"Model: {provider.model_name}")
print(f"Dimension: {provider.dimension}")
print(f"Backend: {provider.backend}")

# Generate embeddings
embeddings = provider.embed(texts, show_progress=True)

print(f"\nEmbeddings shape: {embeddings.shape}")

## 3. Compute Similarities

In [ ]:
# Compute pairwise similarities
from token_emb.embedding_utils import cosine_similarity_matrix

similarity_matrix = cosine_similarity_matrix(embeddings)

# Display as heatmap
plt.figure(figsize=(10, 8))
plt.imshow(similarity_matrix, cmap='viridis', vmin=-1, vmax=1)
plt.colorbar(label='Cosine Similarity')
plt.title('Text Similarity Matrix')
plt.xlabel('Text Index')
plt.ylabel('Text Index')

# Add text annotations
for i in range(len(texts)):
    for j in range(len(texts)):
        plt.text(j, i, f'{similarity_matrix[i, j]:.2f}', 
                ha='center', va='center', color='white', fontsize=8)

plt.tight_layout()
plt.show()

# Show most similar pairs
print("\nMost similar pairs:")
for i in range(len(texts)):
    for j in range(i+1, len(texts)):
        sim = similarity_matrix[i, j]
        print(f"  {sim:.3f} | Text {i} <-> Text {j}")

## 4. Visualize with PCA

In [ ]:
# 2D PCA visualization
visualizer = PCAVisualizer(n_components=2)

fig = visualizer.plot(
    embeddings,
    labels=[f"T{i}" for i in range(len(texts))],
    colors=range(len(texts)),  # Color by index
    title="Text Embeddings - 2D PCA",
    s=200,
    alpha=0.7,
    cmap='tab10'
)

print("\nExplained variance:")
for i, ratio in enumerate(visualizer.explained_variance_ratio):
    print(f"  PC{i+1}: {ratio:.2%}")

In [ ]:
# 3D PCA visualization
visualizer_3d = PCAVisualizer(n_components=3)

fig_3d = visualizer_3d.plot(
    embeddings,
    labels=[f"T{i}" for i in range(len(texts))],
    colors=range(len(texts)),
    title="Text Embeddings - 3D PCA",
    s=200,
    alpha=0.7,
    cmap='tab10'
)

print("\nExplained variance (3D):")
for i, ratio in enumerate(visualizer_3d.explained_variance_ratio):
    print(f"  PC{i+1}: {ratio:.2%}")
print(f"Total: {sum(visualizer_3d.explained_variance_ratio):.2%}")

## 5. Working with Larger Datasets

In [ ]:
# Generate more diverse texts
categories = {
    "animals": [
        "The cat sat on the mat.",
        "Dogs are loyal companions.",
        "Elephants have excellent memory.",
    ],
    "technology": [
        "Python is a popular programming language.",
        "Neural networks are inspired by the brain.",
        "Cloud computing enables scalable infrastructure.",
    ],
    "food": [
        "Pizza is a beloved Italian dish.",
        "Sushi originated in Japan.",
        "Chocolate comes from cacao beans.",
    ],
}

all_texts = []
all_labels = []
all_categories = []

for category, texts_list in categories.items():
    for text in texts_list:
        all_texts.append(text)
        all_labels.append(text[:20] + "...")
        all_categories.append(category)

# Generate embeddings
print(f"Generating embeddings for {len(all_texts)} texts...")
embeddings_cat = provider.embed(all_texts, show_progress=True)

# Create category color mapping
category_colors = {cat: i for i, cat in enumerate(categories.keys())}
colors = [category_colors[cat] for cat in all_categories]

# Visualize
fig = visualizer.plot(
    embeddings_cat,
    labels=[c[0].upper() for c in all_categories],  # First letter as label
    colors=colors,
    title="Texts by Category - PCA Visualization",
    s=200,
    alpha=0.7,
    cmap='Set1'
)

print("\nCategories:")
for cat, color_val in category_colors.items():
    print(f"  {cat}: color {color_val}")

## 6. Save and Load Embeddings

In [ ]:
# Save embeddings
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

np.save(output_dir / "embeddings.npy", embeddings_cat)

# Save texts alongside
with open(output_dir / "texts.txt", "w") as f:
    for text in all_texts:
        f.write(text + "\n")

# Save categories
with open(output_dir / "categories.txt", "w") as f:
    for cat in all_categories:
        f.write(cat + "\n")

print(f"Saved to {output_dir}/")
print("  - embeddings.npy")
print("  - texts.txt")
print("  - categories.txt")

# Load and verify
loaded_embeddings = np.load(output_dir / "embeddings.npy")
print(f"\nLoaded embeddings shape: {loaded_embeddings.shape}")
print(f"Match original: {np.allclose(embeddings_cat, loaded_embeddings)}")

## Next Steps

- Try different embedding models (see `configs/model_paths.yaml`)
- Use the CLI scripts in `scripts/` for batch processing
- Explore more texts and categories
- Try with your own data!